In [1]:
!pip install transformers
!pip install torch
!pip install pillow
!pip install gradio
!pip install torchvision

In [2]:
import torch
from PIL import Image
from transformers import BlipProcessor, BlipForConditionalGeneration
import gradio as gr
import logging
from datetime import datetime

In [3]:
processor = BlipProcessor.from_pretrained(
    "Salesforce/blip-image-captioning-base"
)

model = BlipForConditionalGeneration.from_pretrained(
    "Salesforce/blip-image-captioning-base"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

The image processor of type `BlipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/506 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie text_decoder.cls.predictions.bias to text_decoder.cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BlipForConditionalGeneration LOAD REPORT from: Salesforce/blip-image-captioning-base
Key                                       | Status     |  | 
------------------------------------------+------------+--+-
text_decoder.bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identic

In [4]:
logging.basicConfig(
    filename="caption_logs.txt",
    level=logging.INFO,
    format="%(asctime)s - %(message)s"
)

def log_event(message):
    logging.info(message)

In [5]:
def preprocess_image(image):

    # Convert to RGB
    image = image.convert("RGB")

    # Resize for model
    image = image.resize((384,384))

    return image

In [6]:
def generate_caption(image):

    image = preprocess_image(image)

    inputs = processor(image, return_tensors="pt")

    output = model.generate(**inputs)

    caption = processor.decode(output[0], skip_special_tokens=True)

    log_event("Caption generated")

    return caption

In [13]:
import gradio as gr
from PIL import Image

# Function to generate caption and return image + caption
def caption_app(image):

    caption = generate_caption(image)

    return image, caption


with gr.Blocks(theme=gr.themes.Soft()) as demo:

    gr.Markdown("# AI Image Caption Generator")
    gr.Markdown("Upload an image and the AI will generate a caption.")

    with gr.Row():

        with gr.Column():
            image_input = gr.Image(type="pil", label="Upload Image")

            generate_btn = gr.Button("Generate Caption")

        with gr.Column():

            caption_output = gr.Textbox(
                label="Generated Caption",
                placeholder="Caption will appear here...",
                lines=3
            )

    generate_btn.click(
        fn=caption_app,
        inputs=image_input,
        outputs=[image_output, caption_output],
        show_progress=True
    )

demo.launch()

/tmp/ipykernel_275/1549637038.py:12: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f99a9f1cd74cd44a80.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
